# Customer Intelligence — Integração com Inteligência Artificial

## Objetivo

Integrar Inteligência Artificial Generativa ao projeto Customer Intelligence, utilizando os resultados da segmentação de clientes realizada com RFM e K-Means para transformar dados em informações estratégicas para o negócio.

Nesta etapa, foi desenvolvida uma integração com um modelo de linguagem (LLM) por meio de API, permitindo que a IA interprete os indicadores e características de cada cluster, gere personas representativas dos segmentos de clientes e proponha estratégias de negócio personalizadas.

A partir dessas análises, a IA é capaz de identificar oportunidades de retenção, sugerir ações para clientes em risco de inatividade, recomendar estratégias para aumentar o valor dos consumidores e elaborar sugestões de campanhas e ofertas direcionadas.

Os resultados gerados são estruturados e armazenados em arquivos JSON, permitindo sua reutilização em futuras aplicações.

Essa integração adiciona uma camada de inteligência ao processo de análise, conectando Machine Learning e IA Generativa para transformar a segmentação de clientes em recomendações práticas e apoiar a tomada de decisões estratégicas.

In [1]:
# Manipulação de dados
import pandas as pd

# Arquivos e caminhos
import json
import time
import os
from pathlib import Path

# Modelos e artefatos
import joblib

# Integração com IA
from groq import Groq
from dotenv import load_dotenv, find_dotenv

In [2]:


# 1. Identificar a raiz do projeto
PROJECT_ROOT = Path.cwd().parent

# 2. Definir os caminhos dos artefatos
DATA_PATH = PROJECT_ROOT / "data" / "processed"
MODEL_PATH = PROJECT_ROOT / "models"

# 3. Carregar a base de clientes segmentada
rfm = pd.read_csv(DATA_PATH / "rfm_segmented_customers.csv")

# 4. Carregar os modelos treinados
# scaler = joblib.load(MODEL_PATH / "scaler.pkl")
# kmeans = joblib.load(MODEL_PATH / "kmeans_model.pkl")
mapa_segmentos = joblib.load(MODEL_PATH / "mapa_segmentos.pkl")

print("--- Artefatos carregados com sucesso! ---")
print(f"Total de clientes na base: {len(rfm)}")
print(f"Segmentos disponíveis: {list(mapa_segmentos.values())}")

--- Artefatos carregados com sucesso! ---
Total de clientes na base: 4338
Segmentos disponíveis: ['Clientes Promissores', 'Clientes VIP', 'Clientes Regulares', 'Clientes Inativos']


In [3]:
print("Dimensões da base:", rfm.shape)

display(rfm.head())

print("\nColunas disponíveis:")
print(rfm.columns.tolist())

print("\nDistribuição dos segmentos:")
print(rfm["Segmento"].value_counts())

Dimensões da base: (4338, 6)


,CustomerID,Recency,Frequency,Monetary,Cluster,Segmento
0,12346.0,326,1,77183.60,3,Clientes Promissores
1,12347.0,2,7,4310.00,0,Clientes VIP
2,12348.0,75,4,1797.24,3,Clientes Promissores
3,12349.0,19,1,1757.55,2,Clientes Regulares
4,12350.0,310,1,334.40,1,Clientes Inativos



Colunas disponíveis:
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'Segmento']

Distribuição dos segmentos:
Segmento
Clientes Inativos       1622
Clientes Promissores    1166
Clientes Regulares       837
Clientes VIP             713
Name: count, dtype: int64


In [7]:
# Localiza o arquivo .env na pasta notebooks
env_path = Path.cwd() / ".env"

load_dotenv(env_path, override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("Chave da API carregada com sucesso!")
else:
    print("Chave não encontrada. Verifique o arquivo .env.")

# Inicializa o cliente da Groq
client = Groq(api_key=GROQ_API_KEY)

print("Conexão com a Groq configurada com sucesso!")

Chave da API carregada com sucesso!
Conexão com a Groq configurada com sucesso!


In [ ]:
def construir_prompt(dados_cluster):
    """Constrói o prompt para gerar uma persona e estratégias de CRM."""

    return f"""
    Você é um especialista sênior em CRM e Marketing de E-commerce.
    Analise os dados estatísticos do seguinte segmento de clientes e gere a persona e estratégias.

      IMPORTANTE:
    - Os valores monetários estão em libras esterlinas (GBP).
    - Não converta os valores para reais nem utilize outra moeda.

    DEFINIÇÃO DAS MÉTRICAS:
    - Recency: quantidade de dias desde a última compra.
      Não representa o intervalo médio entre compras.
    - Frequency: quantidade de compras realizadas pelo cliente.
    - Monetary: valor total gasto pelo cliente no período analisado.
      Não representa o ticket médio por pedido.

    REGRAS:
    - Não invente informações que não estejam nos dados.
    - Não confunda gasto total com ticket médio.
    - Não confunda recência com frequência ou intervalo entre compras.
    - Utilize exclusivamente libras esterlinas (£) para valores monetários.
    - Baseie as estratégias nas características observadas de cada segmento.

    Dados do Segmento:
    {json.dumps(dados_cluster, ensure_ascii=False)}

    Retorne a resposta ESTRITAMENTE em formato JSON com as seguintes chaves:
    - "persona_nome": um título marcante para este perfil
    - "descricao_persona": breve resumo comportamental
    - "estrategia_retencao": ação recomendada de marketing
    - "copy_notificacao_push": texto direto para enviar no celular do cliente
    """

In [ ]:
def gerar_resposta_groq(prompt, max_tentativas=3):
    """Envia o prompt à Groq e retorna a resposta convertida em JSON."""

    for tentativa in range(max_tentativas):
        try:
            resposta = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": "Você é um assistente especialista em CRM que responde ESTRITAMENTE em JSON."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                model="openai/gpt-oss-120b",
                temperature=0.2,
                response_format={"type": "json_object"}
            )

            resposta_texto = resposta.choices[0].message.content

            return json.loads(resposta_texto)

        except Exception as erro:
            print(
                f"[Aviso] Falha na tentativa "
                f"{tentativa + 1}/{max_tentativas}: {erro}"
            )

            if tentativa < max_tentativas - 1:
                time.sleep(3)
            else:
                raise

# 1. Resumo estatístico dos 4 clusters a partir da base RFM
resumo_clusters = (
    rfm.groupby(["Cluster", "Segmento"], as_index=False)
    .agg(
        Quantidade_Clientes=("CustomerID", "count"),
        Recency=("Recency", "mean"),
        Frequency=("Frequency", "mean"),
        Monetary=("Monetary", "mean")
    )
)

display(resumo_clusters)

In [16]:
# Geração das personas por segmento
todas_as_personas = {}

# 2. Loop de geração de personas via Groq
for _, row in resumo_clusters.iterrows():
    nome_segmento = row["Segmento"]

    print(f"\nProcessando IA via Groq para: {nome_segmento}...")

    dados_cluster = {
        "segmento": nome_segmento,
        "quantidade_clientes": int(row["Quantidade_Clientes"]),
        "recencia_media_dias": round(row["Recency"], 2),
        "frequencia_media_compras": round(row["Frequency"], 2),
        "valor_monetario_medio_gbp": round(row["Monetary"], 2)
    }

    prompt = construir_prompt(dados_cluster)

    todas_as_personas[nome_segmento] = gerar_resposta_groq(prompt)

    print(f"✓ Persona gerada com sucesso para {nome_segmento}!")

print("\n--- Processamento concluído com sucesso com a Groq! ---")


Processando IA via Groq para: Clientes VIP...
✓ Persona gerada com sucesso para Clientes VIP!

Processando IA via Groq para: Clientes Inativos...
✓ Persona gerada com sucesso para Clientes Inativos!

Processando IA via Groq para: Clientes Regulares...
✓ Persona gerada com sucesso para Clientes Regulares!

Processando IA via Groq para: Clientes Promissores...
✓ Persona gerada com sucesso para Clientes Promissores!

--- Processamento concluído com sucesso com a Groq! ---


In [ ]:
# 3. Exportação dos resultados

personas_path = DATA_PATH / "personas.json"
resumo_path = DATA_PATH / "resumo_clusters_rfm.csv"

# Salvar personas em JSON
with open(personas_path, "w", encoding="utf-8") as arquivo:
    json.dump(todas_as_personas, arquivo, ensure_ascii=False, indent=4)

# Salvar resumo dos clusters em CSV
resumo_clusters.to_csv(resumo_path, index=False)

# Validar o JSON exportado
with open(personas_path, "r", encoding="utf-8") as arquivo:
    personas_salvas = json.load(arquivo)

print("--- Exportação concluída com sucesso! ---")
print(f"Total de personas: {len(personas_salvas)}")
print(f"Segmentos: {list(personas_salvas.keys())}")
print(f"Arquivos salvos em: {DATA_PATH}")

# Exibir as personas geradas
display(personas_salvas)

--- Exportação concluída com sucesso! ---
Total de personas: 4
Segmentos: ['Clientes VIP', 'Clientes Inativos', 'Clientes Regulares', 'Clientes Promissores']
Arquivos salvos em: c:\Users\pedro\OneDrive\projetos\customer-intelligence-ml\data\processed


{'Clientes VIP': {'persona_nome': 'Magnata Digital',
  'descricao_persona': 'Cliente de alto valor, compra frequentemente (≈14 vezes ao ano) e visita o site a cada 12 dias, busca exclusividade e tratamento premium.',
  'estrategia_retencao': 'Implementar programa de fidelidade VIP com benefícios como acesso antecipado a lançamentos, concierge pessoal, descontos exclusivos e eventos privados, reforçando o valor percebido e incentivando compras recorrentes.',
  'copy_notificacao_push': 'Olá! Seu acesso VIP está pronto: pré‑venda exclusiva com 20% de desconto só para você. Aproveite agora!'},
 'Clientes Inativos': {'persona_nome': 'Ex-Comprador Esquecido',
  'descricao_persona': 'Cliente que está inativo há cerca de 180 dias, realiza poucas compras (≈1,3 por período) mas tem ticket médio alto (£341). Possui potencial de retorno se receber um incentivo relevante.',
  'estrategia_retencao': 'Campanha de reengajamento multicanal com oferta exclusiva de desconto ou benefício adicional, segmen

: 